In [1]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq                         
import os

llm = ChatGroq(model="llama-3.3-70b-versatile")

c:\Users\Dhruv\OneDrive\Desktop\ai fundamentals class\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# from langchain_community.tools import DuckDuckGoSearchRun

# search = DuckDuckGoSearchRun()

# search.invoke("what is the capital of france?")

In [2]:
from langchain.tools import tool

In [3]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about current events or general knowledge. """

    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)

    return response

In [4]:
tool_duckduckgo_search.invoke("What is the capital of France?")

'... of these cities is the capital, keep reading to find out. ... What is the capital city of France? The capital, and largest, city of France is Paris. Besides Paris, what is the capital of France? ... You use it between your head and your toes, the more it works the thinner it grows. What is the Capital of France? Paris ... Paris, the capital city of France, is one of the most famous and influential cities in the world. What is the Capital of France? ... As the capital city of France, the city plays host to the national government of France. However, Paris only became the official capital of France during the reign of Clovis I, in the late 5th and early 6th century.'

In [5]:
@tool 
def tool_wikipedia_search(query: str) -> str:
    """Use this tool when you need to answer questions about persons, places, etc."""

    import wikipedia  # ← add this
    wikipedia.set_user_agent("MyLangChainApp/1.0 (your-email@example.com)")  # ← and this

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia_tool.invoke(query)

    return response

In [6]:
tool_wikipedia_search.invoke("Barak Obama")

"Page: Barack Obama\nSummary: Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.\nBorn in Honolulu, Obama graduated from Columbia University in 1983 with a Bachelor of Arts degree in political science and later worked as a community organizer in Chicago. In 1988, Obama enrolled in Harvard Law School, where he was the first Black president of the Harvard Law Review. He became a civil rights attorney and an academic, teaching constitutional law at the University of Chicago Law School from 1992 to 2004. In 1996, Obama was elected to represent the 13th district in the Illinois Senate, a position he held until 2004, when he successfully ran for the U.S. Senate. In the 2008 presidential ele

In [27]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    return result


In [28]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "Dhruv Patel",
        "age": 25,
        "occupation": "Data analyst"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [29]:
tool_personal_info.invoke("Dhruv Patel")

'Dhruv Patel is 25 years old and works as a Data analyst.'

In [44]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool when you need to answer questions based on NovaSpehere Organization's documentation.""" 

    from langchain_community.vectorstores import Chroma
    from langchain_huggingface import HuggingFaceEmbeddings
    embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

    # Retrieve relevant documents from the vector store
    relevant_docs = chroma_db_con.similarity_search(query, k=4)
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])
    return relevant_docs_content

In [45]:
tool_rag.invoke("At the beginning how many employees does started at Novasphere Organization?")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3534.98it/s]


'Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight.\nNovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.\nThe company also started sharing its knowledge through \nblogs and technical tutorials. This helped NovaSphere Technologies build a stronger \npresence in the data engineering community. During 2023, the organization experienced steady growth.\nThis helped the company reduce employee \nturnover and build a stable team that could handle complex projects more efficiently. By 2024, NovaSphere Technologies had become a well-known name among mid-sized \ncompanies that needed data engineering and

## Bind Tools

In [32]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info,
    tool_rag
]

In [33]:
llm_bind = llm.bind_tools(toolkit)

## ReAct agent

In [34]:
from langchain.agents import create_agent
my_agent = create_agent(llm_bind, toolkit)

In [48]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "when was novaSphere Organization founded?"}]}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3068.52it/s]


{'messages': [HumanMessage(content='when was novaSphere Organization founded?', additional_kwargs={}, response_metadata={}, id='f16f7341-d684-46e7-880e-0cb2af0a5257'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bjgvcd098', 'function': {'arguments': '{"query":"NovaSpehere Organization founding date"}', 'name': 'tool_rag'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 621, 'total_tokens': 642, 'completion_time': 0.061133514, 'completion_tokens_details': None, 'prompt_time': 0.033170347, 'prompt_tokens_details': None, 'queue_time': 0.047828232, 'total_time': 0.094303861}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e45fe-24b2-74c2-97e9-3b36d7654f15-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'NovaSpehere Organization founding date'}, 'id': 'bjgvcd0